# ROGII Wellbore Geology

## Score: 25.494

In [ ]:
from pathlib import Path
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")

TRAIN_DIR = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/train")
TEST_DIR = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/test")
SAMPLE_SUB = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv")

ROLL_WINDOWS = (5, 15, 31)
FORMATION_COLS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
SMOOTH_WINDOW = 7
MAX_DELTA_TVT = 2.5
LGB_PARAMS = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 80,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "verbosity": -1,
    "seed": 42,
}
NUM_BOOST_ROUND = 2000
EARLY_STOPPING = 100
N_FOLDS = 5

In [ ]:
def list_wells(data_dir: Path) -> list[str]:
    wells = sorted({p.name.split("__")[0] for p in data_dir.glob("*__horizontal_well.csv")})
    return wells


def load_horizontal(well: str, data_dir: Path) -> pd.DataFrame:
    path = data_dir / f"{well}__horizontal_well.csv"
    df = pd.read_csv(path)
    df.insert(0, "well", well)
    df["row_idx"] = np.arange(len(df))
    return df


def load_typewell(well: str, data_dir: Path) -> pd.DataFrame:
    path = data_dir / f"{well}__typewell.csv"
    tw = pd.read_csv(path).sort_values("TVT").reset_index(drop=True)
    return tw


def interp_typewell_gr(tvt_values: np.ndarray, tw: pd.DataFrame) -> np.ndarray:
    tvt = tw["TVT"].to_numpy(dtype=float)
    gr = tw["GR"].to_numpy(dtype=float)
    return np.interp(tvt_values, tvt, gr, left=gr[0], right=gr[-1])


def eval_start_index(df: pd.DataFrame) -> int:
    need_pred = df["TVT_input"].isna()
    if need_pred.any():
        return int(need_pred.idxmax())
    if df["GR"].notna().any():
        return int(df["GR"].notna().idxmax())
    return len(df)


FORMATION_FEATURE_COLS = [f"tvt_off_{c.lower()}" for c in FORMATION_COLS]


def add_formation_features(df: pd.DataFrame, h: pd.DataFrame, formation_fill: dict[str, float]) -> pd.DataFrame:
    ref = df["tvt_linear"].astype(float)
    for col in FORMATION_COLS:
        fname = f"tvt_off_{col.lower()}"
        if col in h.columns:
            df[fname] = ref - h[col].astype(float)
        else:
            df[fname] = formation_fill.get(fname, 0.0)
    return df


def build_features(
    h: pd.DataFrame,
    tw: pd.DataFrame,
    simulate_eval: bool = False,
    formation_fill: dict[str, float] | None = None,
) -> pd.DataFrame:
    if formation_fill is None:
        formation_fill = {}
    df = h.copy()
    eval_start = eval_start_index(df)
    df["eval_start"] = eval_start
    df["in_eval"] = df["row_idx"] >= eval_start

    tvt_in = df["TVT_input"].copy()
    if simulate_eval and not tvt_in.isna().all():
        tvt_in.iloc[eval_start:] = np.nan

    prev_tvt = tvt_in.shift(1)
    df["tvt_anchor"] = prev_tvt.ffill()
    anchor_md = df["MD"].shift(1).where(prev_tvt.notna()).ffill()
    df["md_anchor"] = anchor_md
    df["md_since_anchor"] = df["MD"] - df["md_anchor"]
    df["tvt_linear"] = df["tvt_anchor"] + df["md_since_anchor"]
    df["ft_into_eval"] = np.maximum(0, df["row_idx"] - eval_start)

    df["d_md"] = df["MD"].diff().fillna(0)
    df["d_z"] = df["Z"].diff().fillna(0)

    gr = df["GR"].astype(float)
    gr_filled = gr.ffill()
    for w in ROLL_WINDOWS:
        df[f"gr_roll_mean_{w}"] = gr_filled.rolling(w, min_periods=1).mean()
        df[f"gr_roll_std_{w}"] = gr_filled.rolling(w, min_periods=1).std().fillna(0)
    df["gr_missing"] = gr.isna().astype(np.int8)

    df = add_formation_features(df, h, formation_fill)

    tvt_lin = df["tvt_linear"].to_numpy(dtype=float)
    tw_gr = interp_typewell_gr(tvt_lin, tw)
    df["tw_gr_at_linear"] = tw_gr
    df["gr_minus_tw"] = gr_filled - tw_gr

    if "TVT" in df.columns:
        df["delta_tvt"] = df["TVT"] - df.groupby("well")["TVT"].shift(1)

    return df


FEATURE_COLS = [
    "MD", "X", "Y", "Z", "d_md", "d_z",
    "tvt_anchor", "md_since_anchor", "tvt_linear", "ft_into_eval",
    "GR", "gr_missing",
    "tw_gr_at_linear", "gr_minus_tw",
    *FORMATION_FEATURE_COLS,
    *[f"gr_roll_mean_{w}" for w in ROLL_WINDOWS],
    *[f"gr_roll_std_{w}" for w in ROLL_WINDOWS],
]


def featurize_well(
    well: str,
    data_dir: Path,
    simulate_eval: bool = False,
    formation_fill: dict[str, float] | None = None,
    h: pd.DataFrame | None = None,
) -> pd.DataFrame:
    if h is None:
        h = load_horizontal(well, data_dir)
    tw = load_typewell(well, data_dir)
    return build_features(h, tw, simulate_eval=simulate_eval, formation_fill=formation_fill)


def deltas_to_tvt(sub: pd.DataFrame, delta_pred: np.ndarray) -> np.ndarray:
    sub = sub.sort_values("row_idx")
    anchor = sub["tvt_anchor"].iloc[0]
    return anchor + np.cumsum(delta_pred)


def smooth_well_tvt(tvt: np.ndarray, max_delta: float = MAX_DELTA_TVT) -> np.ndarray:
    out = pd.Series(tvt).rolling(SMOOTH_WINDOW, center=True, min_periods=1).median().to_numpy()
    for i in range(1, len(out)):
        d = float(np.clip(out[i] - out[i - 1], -max_delta, max_delta))
        out[i] = out[i - 1] + d
    return out

In [ ]:
def build_training_frame(
    wells: list[str],
    data_dir: Path,
    formation_fill: dict[str, float] | None = None,
) -> pd.DataFrame:
    parts = []
    for well in wells:
        df = featurize_well(well, data_dir, simulate_eval=True, formation_fill=formation_fill)
        if "TVT" not in df.columns:
            continue
        mask = (
            df["in_eval"]
            & df["GR"].notna()
            & df["TVT"].notna()
            & df["tvt_anchor"].notna()
            & df["delta_tvt"].notna()
        )
        parts.append(df.loc[mask])
    return pd.concat(parts, ignore_index=True)


train_wells = list_wells(TRAIN_DIR)
print(f"Training wells: {len(train_wells)}")

train_df = build_training_frame(train_wells, TRAIN_DIR, formation_fill={})
FORMATION_FILL = {c: float(train_df[c].median()) for c in FORMATION_FEATURE_COLS}
MAX_DELTA_TVT = float(np.percentile(np.abs(train_df["delta_tvt"]), 95))
print("Formation fill (test impute):", {k: round(v, 2) for k, v in FORMATION_FILL.items()})
print(f"MAX_DELTA_TVT: {MAX_DELTA_TVT:.3f}")

print(f"Training rows (eval zone only): {len(train_df):,}")
print(train_df[["TVT", "delta_tvt", "tvt_linear", "GR"]].describe().round(3))

In [ ]:
X = train_df[FEATURE_COLS]
y = train_df["delta_tvt"].astype(float)
groups = train_df["well"]

oof_tvt = np.zeros(len(train_df))
models = []

gkf = GroupKFold(n_splits=N_FOLDS)
for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups=groups)):
    dtrain = lgb.Dataset(X.iloc[tr_idx], label=y.iloc[tr_idx])
    dvalid = lgb.Dataset(X.iloc[va_idx], label=y.iloc[va_idx], reference=dtrain)

    model = lgb.train(
        LGB_PARAMS,
        dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[dvalid],
        valid_names=["valid"],
        callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False)],
    )
    models.append(model)
    va = train_df.iloc[va_idx]
    delta_pred = model.predict(X.iloc[va_idx], num_iteration=model.best_iteration)

    for well in va["well"].unique():
        well_mask = (va["well"] == well).to_numpy()
        wpos = va_idx[well_mask]
        oof_tvt[wpos] = deltas_to_tvt(train_df.iloc[wpos], delta_pred[well_mask])

    fold_rmse = np.sqrt(mean_squared_error(train_df.iloc[va_idx]["TVT"], oof_tvt[va_idx]))
    print(f"Fold {fold + 1}/{N_FOLDS}  eval-zone TVT RMSE = {fold_rmse:.4f}  (best_iter={model.best_iteration})")

cv_rmse = np.sqrt(mean_squared_error(train_df["TVT"], oof_tvt))
print(f"\nOOF eval-zone TVT RMSE (GroupKFold by well): {cv_rmse:.4f}")

In [ ]:
def predict_well_tvt(
    h: pd.DataFrame,
    tw: pd.DataFrame,
    models: list,
    formation_fill: dict[str, float],
    eval_row_idx: np.ndarray | None = None,
) -> pd.DataFrame:
    df = build_features(h, tw, simulate_eval=False, formation_fill=formation_fill)
    if eval_row_idx is None:
        need = df["in_eval"] & h["TVT_input"].isna()
    else:
        need = df["row_idx"].isin(eval_row_idx)
    sub = df.loc[need].copy()
    if sub.empty:
        return pd.DataFrame(columns=["id", "tvt", "row_idx"])

    Xp = sub[FEATURE_COLS]
    delta_pred = np.mean([m.predict(Xp, num_iteration=m.best_iteration) for m in models], axis=0)
    sub["tvt"] = deltas_to_tvt(sub, delta_pred)
    sub["id"] = sub["well"] + "_" + sub["row_idx"].astype(str)
    return sub[["id", "tvt", "row_idx"]]


def predict_eval_zone(
    well: str,
    data_dir: Path,
    models: list,
    formation_fill: dict[str, float],
) -> pd.DataFrame:
    h = load_horizontal(well, data_dir)
    tw = load_typewell(well, data_dir)

    pass1 = predict_well_tvt(h, tw, models, formation_fill)
    if pass1.empty:
        return pd.DataFrame(columns=["id", "tvt"])

    eval_row_idx = pass1["row_idx"].to_numpy()
    h2 = h.copy()
    for _, row in pass1.iterrows():
        h2.loc[h2["row_idx"] == row["row_idx"], "TVT_input"] = row["tvt"]

    pass2 = predict_well_tvt(h2, tw, models, formation_fill, eval_row_idx=eval_row_idx)
    tvt = smooth_well_tvt(pass2["tvt"].to_numpy())
    pass2["tvt"] = tvt
    return pass2[["id", "tvt"]]


test_wells = list_wells(TEST_DIR)
print(f"Test wells: {len(test_wells)}")

pred_parts = [predict_eval_zone(w, TEST_DIR, models, FORMATION_FILL) for w in test_wells]
submission = pd.concat(pred_parts, ignore_index=True)

sample = pd.read_csv(SAMPLE_SUB)
submission = sample[["id"]].merge(submission, on="id", how="left")
missing = submission["tvt"].isna().sum()
if missing:
    raise ValueError(f"Missing predictions for {missing} sample ids")

submission.to_csv("submission.csv", index=False)
print(submission.head())
print(f"\nWrote submission.csv  rows={len(submission):,}")
print(f"tvt range: {submission['tvt'].min():.2f} .. {submission['tvt'].max():.2f}")